# ML Tutor — Week 4: Classification, imbalance & metrics
**Track:** Core ML · **Lab:** Lab 4 — ADR prediction when positives are rare

**Objectives this week:**
1. Compare classifier families (logistic regression, trees, KNN) on one task. *(CO2, CO5)*
2. Handle class imbalance with class weights and resampling. *(CO5)*
3. Select and interpret precision, recall, AUROC, and calibration for ADR prediction. *(CO5)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


## Before you start (~25 min)
**Goal for this session:** Arrive ready to compare model outputs using the Week 2 train/test habit and the Week 3 data-cleaning mindset. You do not need to memorize formulas; you do need to remember what a label is and why a split must stay honest. Reference delta: make confusion-matrix reading mandatory and keep calibration detail optional.

**Warm up — answer these from memory before scrolling on** (retrieval beats re-reading):
1. From Week 2, what is the difference between a feature column and a label column?
2. From Week 3, name two data-quality problems that could quietly damage a model before it ever trains.
3. If a model predicts a rare event, why might plain accuracy sound better than the model really is?


In [ ]:
# SETUP — run me first (Shift+Enter)
import numpy as np
import pandas as pd

# Dataset: Synthetic medication-exposure table with columns such as age_band, med_class, dose_band, renal_flag, prior_adr, polypharmacy_count, and adr_flag. Rare-event binary label; no PHI; course-generated teaching data.
# PLACEHOLDER: the dataset for this week arrives with the course repo under data/week-04/ .
# If a lab step expects a file, download it from the ml-tutor-labs repo's data/week-04/ folder first.

print("Setup OK — numpy", np.__version__, "· pandas", pd.__version__)


## Worked example — Classifier families on one task
**Lane A · the general idea:** Logistic regression draws a weighted boundary, trees split by rules, and KNN votes from nearby examples. They can all solve the same binary task but fail in different ways.

**Lane B · the same idea in pharmacy terms:** For ADR prediction, logistic regression may offer smoother probability scores, trees may expose human-readable splits, and KNN may struggle if the feature space mixes unlike patients.

Below is a tiny, complete demo of this idea. Run it, read every line, change one thing, run it again.


In [ ]:
# WORKED EXAMPLE — accuracy lies on imbalanced data (ADR prediction)
from sklearn.metrics import accuracy_score, recall_score

# 20 patients, only 2 true ADRs — realistic imbalance, synthetic labels
y_true = [0]*18 + [1]*2
y_lazy = [0]*20                    # a "model" that never flags an ADR

print("accuracy:", accuracy_score(y_true, y_lazy))   # 0.9 — looks great
print("recall  :", recall_score(y_true, y_lazy))     # 0.0 — misses EVERY ADR
# same predictions, two metrics, opposite stories. Pick metrics for the harm that matters.


## 1. Baseline the classification task
Load the prepared ADR dataframe, inspect class balance, and fit one baseline classifier with a clean train/test split.

**Checkpoint:** Checkpoint 1 verifies: X and y are separated correctly, the split uses stratify=y, and the baseline pipeline fits without using test data during preprocessing.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# adr_df is already loaded in the notebook (synthetic, teaching-only; inline here too).
adr_records = [
    {"age_band": "18-40", "med_class": "antibiotic",  "dose_band": "low",  "renal_flag": 0, "prior_adr": 0, "polypharmacy_count": 1, "prior_ed_visits": 0, "renal_score": 0.9, "adr_flag": 0},
    {"age_band": "41-64", "med_class": "anticoagulant","dose_band": "high", "renal_flag": 1, "prior_adr": 1, "polypharmacy_count": 5, "prior_ed_visits": 2, "renal_score": 0.4, "adr_flag": 1},
    {"age_band": "65+",   "med_class": "antibiotic",  "dose_band": "med",  "renal_flag": 1, "prior_adr": 0, "polypharmacy_count": 4, "prior_ed_visits": 1, "renal_score": 0.5, "adr_flag": 0},
    {"age_band": "18-40", "med_class": "analgesic",   "dose_band": "low",  "renal_flag": 0, "prior_adr": 0, "polypharmacy_count": 0, "prior_ed_visits": 0, "renal_score": 1.0, "adr_flag": 0},
    {"age_band": "65+",   "med_class": "anticoagulant","dose_band": "high", "renal_flag": 1, "prior_adr": 1, "polypharmacy_count": 6, "prior_ed_visits": 3, "renal_score": 0.3, "adr_flag": 1},
    {"age_band": "41-64", "med_class": "analgesic",   "dose_band": "med",  "renal_flag": 0, "prior_adr": 0, "polypharmacy_count": 2, "prior_ed_visits": 0, "renal_score": 0.8, "adr_flag": 0},
    {"age_band": "18-40", "med_class": "antibiotic",  "dose_band": "low",  "renal_flag": 0, "prior_adr": 0, "polypharmacy_count": 1, "prior_ed_visits": 0, "renal_score": 0.95,"adr_flag": 0},
    {"age_band": "65+",   "med_class": "anticoagulant","dose_band": "high", "renal_flag": 1, "prior_adr": 1, "polypharmacy_count": 7, "prior_ed_visits": 2, "renal_score": 0.35,"adr_flag": 1},
    {"age_band": "41-64", "med_class": "analgesic",   "dose_band": "low",  "renal_flag": 0, "prior_adr": 0, "polypharmacy_count": 1, "prior_ed_visits": 0, "renal_score": 0.85,"adr_flag": 0},
    {"age_band": "18-40", "med_class": "antibiotic",  "dose_band": "med",  "renal_flag": 0, "prior_adr": 0, "polypharmacy_count": 2, "prior_ed_visits": 0, "renal_score": 0.9, "adr_flag": 0},
]
adr_df = pd.DataFrame(adr_records)
target = "adr_flag"

# TODO: separate X and y
X = ...
y = ...

# TODO: make an honest split with stratify=y
X_train, X_test, y_train, y_test = ...

categorical_cols = ["age_band", "med_class", "dose_band"]
numeric_cols = ["polypharmacy_count", "prior_ed_visits", "renal_score"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_cols),
    ]
)

baseline_model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

# TODO: fit the baseline model
...


**Self-check 1:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 2. Read the confusion matrix before celebrating
Generate test-set predictions and compute accuracy, precision, recall, and the confusion matrix for the baseline.

**Checkpoint:** Checkpoint 2 verifies: all four summaries are computed from test-set predictions, and the notebook includes a one-sentence interpretation of which error type is currently dominating.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

# TODO: generate hard predictions on X_test
y_pred = ...

# TODO: compute the four baseline summaries
acc = ...
prec = ...
rec = ...
cm = ...

print("accuracy:", acc)
print("precision:", prec)
print("recall:", rec)
print("confusion matrix:\n", cm)


**Self-check 2:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 3. Address imbalance without peeking at the test set
Try one imbalance-aware adjustment, either class weights or a cautious resampling strategy, and compare it with the baseline.

**Checkpoint:** Checkpoint 3 verifies: the adjustment is applied only to training-time behavior, not the full dataset; students report at least one metric that improved and one tradeoff that appeared.


In [ ]:
# Option A: class weights inside logistic regression
weighted_model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight=...)),
])

# Option B (if provided in the notebook): a resampling cell that acts on TRAIN only.
# Do not resample the full dataset before splitting.

# TODO: fit your adjusted model and compute test metrics again
...


**Self-check 3:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 4. Add probability thinking: AUROC and calibration check
Use predicted probabilities to compute AUROC and inspect whether the model scores sound overconfident or underconfident.

**Checkpoint:** Checkpoint 4 verifies: y_score comes from predict_proba for the positive class, AUROC is computed correctly, and the reflection distinguishes ranking quality from calibrated probability meaning.


In [ ]:
from sklearn.metrics import roc_auc_score

# TODO: get the positive-class probabilities from your chosen model
y_score = ...

auc = ...
print("auroc:", auc)

# Write 2-3 sentences:
# 1) What does AUROC say about ranking?
# 2) Does that automatically mean the probabilities are well calibrated?
# 3) For this ADR task, which metric would you report first and why?


**Self-check 4:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## Reflection — make it stick (5 min, write before you leave)
1. **Teach it:** explain your Step 4 code to a colleague in exactly 3 sentences — what goes in, what happens, what comes out. Write the sentences below.
2. **What would break this?** A classmate insists: *“High accuracy means the classifier is good.”* Using what you just built, describe one concrete case from this week's lab where acting on that belief produces a wrong result — and how you would catch it.


In [ ]:
# Your reflection (write as comments)
# 1) Three sentences:
#
# 2) What would break this, concretely:
#



## Optional challenge — Homework: HW4 — Metric memo for an ADR model
Train two beginner-friendly classifiers on the teaching ADR dataset, then write a short comparison memo. Include class balance, confusion matrices, precision, recall, and AUROC. Choose which model you would carry forward for more study and justify the choice in workflow terms, not by naming only the largest score.

**Deliverable:** One runnable notebook plus a 250- to 350-word memo. The memo must name one remaining limitation, one leakage risk you avoided, and one reason a probability score should not be treated as clinical truth.


In [ ]:
# HW / challenge workspace — build it here

